# Gemma 4 × Crypto Portfolio & Research Agent (Steps 19-20)

A **read-only** crypto intelligence agent: Gemma 4 analyses on-chain data via the
**Blockscout MCP** (blockchain explorer) and searches tokens via the **wallet MCP**,
while you retain full control of signing and sending.

## What you need
- [Ollama](https://ollama.com) running with `gemma4:12b` pulled
- `llama-index-llms-ollama`, `llama-index-tools-mcp` installed
- Node.js ≥18 (for Blockscout MCP via `npx`)
- **Optional**: `WALLET_MCP_URL` pointing at your wallet MCP server
  (only `get_portfolio` and `search_tokens` are used — never `send`/`sign`/`swap`)

```bash
pip install llama-index-llms-ollama llama-index-tools-mcp
ollama pull gemma4:12b
```

> **Security**: This notebook is intentionally read-only. The `send`, `sign`, and
> `swap` wallet tools are **excluded** by filtering. Any transaction requires
> leaving this notebook and using your wallet app directly.

## Part 1 — Blockscout On-Chain Explorer

In [ ]:
import os
import asyncio

from llama_index.llms.ollama import Ollama
from llama_index.tools.mcp import BasicMCPClient, McpToolSpec
from llama_index.core.agent.workflow import ReActAgent

llm = Ollama(model="gemma4:12b", request_timeout=120.0)

# Blockscout MCP — read-only blockchain explorer across 9+ chains
blockscout_client = BasicMCPClient(
    "npx",
    args=["-y", "@blockscout/mcp-server"],
)

blockscout_spec = McpToolSpec(client=blockscout_client)
blockscout_tools = await blockscout_spec.to_tool_list_async()

print(f"Loaded {len(blockscout_tools)} Blockscout tools:")
for t in blockscout_tools:
    print(f"  • {t.metadata.name}")

In [ ]:
# Chain IDs for reference:
# Ethereum: 1 | Polygon: 137 | Base: 8453 | Arbitrum: 42161
# Optimism: 10 | Gnosis: 100 | Celo: 42220

blockscout_agent = ReActAgent(
    tools=blockscout_tools,
    llm=llm,
    max_iterations=15,
    verbose=True,
)

## Part 2 — Wallet Address Analysis

In [ ]:
# Set your wallet address — READ ONLY, never your private key
MY_WALLET = os.environ.get("WALLET_ADDRESS", "0x0000000000000000000000000000000000000000")

if MY_WALLET.startswith("0x000"):
    print("Set WALLET_ADDRESS env var to your public wallet address for live analysis.")

In [ ]:
# Ethereum portfolio overview
response = await blockscout_agent.run(
    f"Get the full balance and token holdings for address {MY_WALLET} on Ethereum (chain_id=1). "
    "Show: ETH balance, ERC-20 token names and amounts, and estimated USD values if available. "
    "Summarise the portfolio allocation as percentages."
)
print(response)

In [ ]:
# Multi-chain portfolio sweep
response = await blockscout_agent.run(
    f"Check address {MY_WALLET} on these chains: Ethereum (1), Base (8453), Polygon (137). "
    "For each chain, report the native token balance and top 3 ERC-20 holdings by value. "
    "Then give a combined cross-chain portfolio summary."
)
print(response)

In [ ]:
# Recent transaction history
response = await blockscout_agent.run(
    f"Get the 10 most recent transactions for {MY_WALLET} on Ethereum. "
    "For each transaction: hash, date, type (send/receive/contract interaction), "
    "counterparty address, value in ETH, and gas paid. "
    "Flag any interactions with known DeFi protocols."
)
print(response)

In [ ]:
# NFT holdings
response = await blockscout_agent.run(
    f"List all NFTs held by {MY_WALLET} on Ethereum. "
    "Group by collection name. Show token IDs and contract addresses."
)
print(response)

## Part 3 — Token Research & Due Diligence

In [ ]:
# Look up a token by symbol and inspect its contract
response = await blockscout_agent.run(
    "Look up the USDC token on Ethereum (chain_id=1). "
    "Get the contract address, verify it's a known legitimate contract, "
    "show the total supply, and fetch the first 20 lines of the contract ABI."
)
print(response)

In [ ]:
# Inspect an unknown contract before interacting with it
UNKNOWN_CONTRACT = "0xA0b86991c6218b36c1d19D4a2e9Eb0cE3606eB48"  # USDC as example
response = await blockscout_agent.run(
    f"Inspect contract {UNKNOWN_CONTRACT} on Ethereum (chain_id=1). "
    "Is it verified? What does it do based on the ABI and code? "
    "Are there any red flags (proxy upgradeable, ownable with centralised admin, mint function)? "
    "Give a 3-sentence safety assessment."
)
print(response)

In [ ]:
# Token transfer history analysis
response = await blockscout_agent.run(
    f"Get all ERC-20 token transfers for {MY_WALLET} on Ethereum in the last 30 days. "
    "Group by token symbol. Show net flow (received vs sent) for each token. "
    "Identify the most active DeFi interactions."
)
print(response)

## Part 4 — Read-Only Wallet Portfolio Tool

If you have the wallet MCP server running, use it for `get_portfolio` and
`search_tokens` only. The dangerous tools (`send`, `sign`, `swap`) are explicitly
filtered out and never loaded.

In [ ]:
WALLET_MCP_URL = os.environ.get("WALLET_MCP_URL", "")

# Read-only wallet tools — the allowlist is the security boundary
READ_ONLY_WALLET_TOOLS = {"get_portfolio", "search_tokens", "get_wallets", "get_transaction_history"}
DANGEROUS_TOOLS = {"send", "sign", "swap", "send_calls"}  # never loaded

wallet_tools = []
if WALLET_MCP_URL:
    wallet_client = BasicMCPClient(WALLET_MCP_URL)
    all_wallet_tools = await McpToolSpec(client=wallet_client).to_tool_list_async()

    # Strict allowlist — only read-only tools loaded
    wallet_tools = [
        t for t in all_wallet_tools
        if t.metadata.name in READ_ONLY_WALLET_TOOLS
    ]

    blocked = [t.metadata.name for t in all_wallet_tools if t.metadata.name in DANGEROUS_TOOLS]
    print(f"Loaded {len(wallet_tools)} read-only wallet tools: {[t.metadata.name for t in wallet_tools]}")
    print(f"Blocked dangerous tools: {blocked}")
else:
    print("WALLET_MCP_URL not set — skipping wallet MCP. Blockscout tools are still available.")

In [ ]:
# Combined research agent: Blockscout (on-chain) + wallet portfolio (if available)
all_read_only_tools = blockscout_tools + wallet_tools

research_agent = ReActAgent(
    tools=all_read_only_tools,
    llm=llm,
    max_iterations=15,
    verbose=True,
)

print(f"Research agent ready with {len(all_read_only_tools)} total read-only tools.")

In [ ]:
# Portfolio summary across both tools
response = await research_agent.run(
    "Give me a complete read-only portfolio overview: "
    "use the wallet portfolio tool (if available) for a high-level summary, "
    f"then use Blockscout to verify the on-chain balances for {MY_WALLET} on Ethereum and Base. "
    "Reconcile any differences and flag discrepancies."
)
print(response)

## Part 5 — Wolfram Alpha + On-Chain Data (Math-Grounded Analysis)

In [ ]:
WOLFRAM_APP_ID = os.environ.get("WOLFRAM_APP_ID", "")

wolfram_tools = []
if WOLFRAM_APP_ID:
    # Wolfram MCP via SSE
    wolfram_client = BasicMCPClient(
        "npx",
        args=["-y", "@modelcontextprotocol/server-wolfram-alpha"],
        env={**os.environ, "WOLFRAM_APP_ID": WOLFRAM_APP_ID},
    )
    wolfram_tools = await McpToolSpec(client=wolfram_client).to_tool_list_async()
    print(f"Wolfram tools loaded: {[t.metadata.name for t in wolfram_tools]}")
else:
    print("WOLFRAM_APP_ID not set — Wolfram tools skipped.")

In [ ]:
if wolfram_tools:
    math_agent = ReActAgent(
        tools=all_read_only_tools + wolfram_tools,
        llm=llm,
        max_iterations=15,
        verbose=True,
    )

    # Compound interest / yield calculations grounded in real balances
    response = await math_agent.run(
        f"Check the ETH balance of {MY_WALLET} on Ethereum using Blockscout. "
        "Then use Wolfram Alpha to calculate: if I staked that ETH at 4.2% APY, "
        "what would the balance be in 1 year, 3 years, and 5 years? Show the maths."
    )
    print(response)

## Part 6 — ENS Name Resolution

In [ ]:
# Resolve an ENS name to an address and analyse it
ENS_NAME = "vitalik.eth"  # change to any ENS name
response = await blockscout_agent.run(
    f"Resolve the ENS name '{ENS_NAME}' to an Ethereum address. "
    "Then get the current ETH balance and the 5 most recent transactions for that address."
)
print(response)

## Part 7 — Block & Network Stats

In [ ]:
# Current block number and network activity
response = await blockscout_agent.run(
    "What is the current block number on Ethereum (chain_id=1)? "
    "Get info on the latest block: timestamp, transaction count, gas used, gas limit. "
    "What percentage of gas capacity was used?"
)
print(response)

In [ ]:
# Inspect a specific transaction
TX_HASH = "0x"  # replace with a real tx hash
if TX_HASH != "0x":
    response = await blockscout_agent.run(
        f"Get full details for transaction {TX_HASH} on Ethereum. "
        "Explain in plain language what this transaction did: "
        "who sent to whom, what value, what contract method was called (if any), "
        "was it successful, and what were the gas costs in ETH and USD?"
    )
    print(response)
else:
    print("Set TX_HASH to a real transaction hash to inspect it.")

## Security reminders

- **Never** put a private key or seed phrase in this notebook or any env var an agent can read
- `WALLET_ADDRESS` is your **public** address — safe to share with blockchain explorers
- The wallet MCP allowlist (`READ_ONLY_WALLET_TOOLS`) is the hard boundary:
  `send`, `sign`, `swap`, `send_calls` are never loaded — not just skipped
- For any transaction: close this notebook, use your hardware wallet or mobile app directly
- Read-only Blockscout tools require no API key and no credentials — purely public data
- Use a view-only API key for any exchange integrations (Coinbase, Bitget read-only key)